# 01 · Manipulación de datos con Pandas y Polars

Antes de entrenar cualquier modelo hay que **cargar, explorar, limpiar y transformar** los datos.
Aquí hacemos cada operación dos veces: una con **Pandas** (el estándar histórico) y otra con **Polars**
(más reciente, escrito en Rust, multihilo y con evaluación *lazy*).

**Objetivos**
1. Cargar un CSV e inspeccionarlo.
2. Detectar y tratar valores faltantes.
3. Filtrar, crear columnas y agrupar.
4. Entender la API de expresiones de Polars y el modo *lazy*.
5. Convertir entre ambas librerías.

> Dataset: `datos/casas.csv` (sintético, generado con `datos/generar_datos.py`). Objetivo a predecir: `precio`.

In [1]:
# Librerias a usar
#=====================================================
import time

import numpy as np
import pandas as pd
import polars as pl

print("pandas", pd.__version__, "| polars", pl.__version__)
RUTA = "datos/casas.csv"

pandas 3.0.6 | polars 1.44.2


## 1. Cargar e inspeccionar

In [2]:
# Pandas
#=====================================================
df_pd = pd.read_csv(RUTA)
df_pd.head()

,id,barrio,metros,habitaciones,antiguedad,distancia_centro_km,tiene_cochera,precio
0,1,Sur,74.1,1,22.0,24.31,si,41500.0
1,2,Norte,141.8,3,8.0,9.23,no,285500.0
2,3,Periferia,67.9,3,32.0,13.96,si,49700.0
3,4,Sur,100.0,2,33.0,13.05,no,77900.0
4,5,Centro,89.5,2,44.0,0.19,no,133400.0


In [3]:
# Polars
#=====================================================
df_pl = pl.read_csv(RUTA)
df_pl.head()

id,barrio,metros,habitaciones,antiguedad,distancia_centro_km,tiene_cochera,precio
i64,str,f64,i64,f64,f64,str,f64
1,"""Sur""",74.1,1,22.0,24.31,"""si""",41500.0
2,"""Norte""",141.8,3,8.0,9.23,"""no""",285500.0
3,"""Periferia""",67.9,3,32.0,13.96,"""si""",49700.0
4,"""Sur""",100.0,2,33.0,13.05,"""no""",77900.0
5,"""Centro""",89.5,2,44.0,0.19,"""no""",133400.0


In [4]:
# Tipos de datos y resumen estadístico
#=====================================================
df_pd.info()
df_pd.describe()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   2000 non-null   int64  
 1   barrio               2000 non-null   str    
 2   metros               1920 non-null   float64
 3   habitaciones         2000 non-null   int64  
 4   antiguedad           1940 non-null   float64
 5   distancia_centro_km  2000 non-null   float64
 6   tiene_cochera        2000 non-null   str    
 7   precio               2000 non-null   float64
dtypes: float64(4), int64(2), str(2)
memory usage: 139.7 KB


,id,metros,habitaciones,antiguedad,distancia_centro_km,precio
count,2000.000000,1920.000000,2000.000000,1940.000000,2000.000000,2000.000000
mean,1000.500000,111.080156,3.194500,29.060825,11.145635,159529.050000
std,577.494589,34.613266,1.283937,17.236558,7.679769,104147.157841
min,1.000000,35.000000,1.000000,0.000000,0.020000,30000.000000
25%,500.750000,87.150000,2.000000,15.000000,3.287500,75300.000000
50%,1000.500000,110.650000,3.000000,29.000000,10.645000,143900.000000
75%,1500.250000,134.250000,4.000000,44.000000,17.842500,224900.000000
max,2000.000000,218.100000,7.000000,59.000000,24.990000,574900.000000


In [5]:
print(df_pl.schema)
df_pl.describe()

Schema([('id', Int64), ('barrio', String), ('metros', Float64), ('habitaciones', Int64), ('antiguedad', Float64), ('distancia_centro_km', Float64), ('tiene_cochera', String), ('precio', Float64)])


statistic,id,barrio,metros,habitaciones,antiguedad,distancia_centro_km,tiene_cochera,precio
str,f64,str,f64,f64,f64,f64,str,f64
"""count""",2000.0,"""2000""",1920.0,2000.0,1940.0,2000.0,"""2000""",2000.0
"""null_count""",0.0,"""0""",80.0,0.0,60.0,0.0,"""0""",0.0
"""mean""",1000.5,null,111.080156,3.1945,29.060825,11.145635,null,159529.05
"""std""",577.494589,null,34.613266,1.283937,17.236558,7.679769,null,104147.157841
"""min""",1.0,"""Centro""",35.0,1.0,0.0,0.02,"""no""",30000.0
"""25%""",501.0,null,87.2,2.0,15.0,3.29,null,75300.0
"""50%""",1001.0,null,110.7,3.0,29.0,10.65,null,144000.0
"""75%""",1500.0,null,134.2,4.0,44.0,17.84,null,224800.0
"""max""",2000.0,"""Sur""",218.1,7.0,59.0,24.99,"""si""",574900.0


## 2. Valores faltantes

Siempre revisa cuántos valores faltan **por columna** antes de modelar.
Pandas usa `NaN`; Polars distingue entre `null` (faltante) y `NaN` (resultado numérico inválido).

In [6]:
print("Pandas:\n", df_pd.isna().sum(), sep="")
print("\nPolars:")
df_pl.null_count()

Pandas:
id                      0
barrio                  0
metros                 80
habitaciones            0
antiguedad             60
distancia_centro_km     0
tiene_cochera           0
precio                  0
dtype: int64

Polars:


id,barrio,metros,habitaciones,antiguedad,distancia_centro_km,tiene_cochera,precio
u32,u32,u32,u32,u32,u32,u32,u32
0,0,80,0,60,0,0,0


In [7]:
# Imputar con la mediana (ojo: en ML real la mediana se calcula SOLO con train, ver notebook 02)
#=====================================================
df_pd_limpio = df_pd.fillna({
    "metros": df_pd["metros"].median(),
    "antiguedad": df_pd["antiguedad"].median(),
})

df_pl_limpio = df_pl.with_columns(
    pl.col("metros").fill_null(pl.col("metros").median()),
    pl.col("antiguedad").fill_null(pl.col("antiguedad").median()),
)
print(df_pd_limpio.isna().sum().sum(), df_pl_limpio.null_count().sum_horizontal().item())

0 0


## 3. Filtrar, crear columnas y ordenar

En Polars todo se expresa con **expresiones** (`pl.col(...)`) dentro de contextos:
`select`, `with_columns`, `filter`, `group_by(...).agg(...)`.

In [8]:
# Pandas: casas del Centro con más de 100 m², con precio por m²
#=====================================================
res_pd = (
    df_pd_limpio
    .assign(precio_m2=lambda d: d["precio"] / d["metros"])
    .query("barrio == 'Centro' and metros > 100")
    .sort_values("precio_m2", ascending=False)
    [["barrio", "metros", "precio", "precio_m2"]]
)
res_pd.head()

,barrio,metros,precio,precio_m2
340,Centro,110.65,387100.0,3498.418437
1296,Centro,108.40,368400.0,3398.523985
85,Centro,142.30,476100.0,3345.748419
434,Centro,108.40,362300.0,3342.250923
1340,Centro,131.70,439600.0,3337.889142


In [9]:
# Polars: lo mismo
#=====================================================
res_pl = (
    df_pl_limpio
    .with_columns((pl.col("precio") / pl.col("metros")).alias("precio_m2"))
    .filter((pl.col("barrio") == "Centro") & (pl.col("metros") > 100))
    .sort("precio_m2", descending=True)
    .select("barrio", "metros", "precio", "precio_m2")
)
res_pl.head()

barrio,metros,precio,precio_m2
str,f64,f64,f64
"""Centro""",110.65,387100.0,3498.418437
"""Centro""",108.4,368400.0,3398.523985
"""Centro""",142.3,476100.0,3345.748419
"""Centro""",108.4,362300.0,3342.250923
"""Centro""",131.7,439600.0,3337.889142


## 4. Agrupar y agregar

In [10]:
# Pandas
#=====================================================
df_pd_limpio.groupby("barrio").agg(
    n=("precio", "size"),
    precio_medio=("precio", "mean"),
    metros_medios=("metros", "mean"),
).sort_values("precio_medio", ascending=False)

,n,precio_medio,metros_medios
barrio,,,
Centro,408,262992.156863,111.455025
Norte,600,159701.000000,110.656500
Sur,579,125191.364421,111.138515
Periferia,413,105207.990315,111.160169


In [11]:
# Polars
#=====================================================
df_pl_limpio.group_by("barrio").agg(
    pl.len().alias("n"),
    pl.col("precio").mean().alias("precio_medio"),
    pl.col("metros").mean().alias("metros_medios"),
).sort("precio_medio", descending=True)

barrio,n,precio_medio,metros_medios
str,u32,f64,f64
"""Centro""",408,262992.156863,111.455025
"""Norte""",600,159701.0,110.6565
"""Sur""",579,125191.364421,111.138515
"""Periferia""",413,105207.990315,111.160169


## 5. Modo *lazy* de Polars

Con `scan_csv` Polars no lee nada hasta que llamas a `.collect()`. Mientras tanto construye un
**plan de consulta** y lo optimiza (p. ej. solo lee las columnas que usas y aplica filtros temprano).
Esto es clave con archivos grandes.

In [12]:
consulta = (
    pl.scan_csv(RUTA)
    .filter(pl.col("tiene_cochera") == "si")
    .group_by("barrio")
    .agg(pl.col("precio").median().alias("mediana_precio"))
    .sort("mediana_precio")
)
print(consulta.explain())   # plan optimizado
consulta.collect()

SORT BY [col("mediana_precio")]
  AGGREGATE[maintain_order: false]
    [col("precio").median().alias("mediana_precio")] BY [col("barrio")]
    FROM
    simple π 2/2 ["barrio", "precio"]
      Csv SCAN [datos/casas.csv]
      PROJECT 3/8 COLUMNS
      SELECTION: col("tiene_cochera") == "si"
      ESTIMATED ROWS: 2296


barrio,mediana_precio
str,f64
"""Periferia""",109000.0
"""Sur""",127200.0
"""Norte""",168550.0
"""Centro""",269000.0


## 6. Comparación rápida de velocidad

Con 2 000 filas la diferencia es mínima; replicamos los datos para notarla.
Los tiempos dependen de tu máquina.

In [13]:
grande_pd = pd.concat([df_pd_limpio] * 500, ignore_index=True)
grande_pl = pl.from_pandas(grande_pd)
print(f"{len(grande_pd):,} filas")

t = time.perf_counter()
grande_pd.groupby("barrio")["precio"].agg(["mean", "std", "median"])
print(f"pandas: {time.perf_counter() - t:.3f} s")

t = time.perf_counter()
grande_pl.group_by("barrio").agg(
    pl.col("precio").mean(), pl.col("precio").std().alias("std"), pl.col("precio").median().alias("median")
)
print(f"polars: {time.perf_counter() - t:.3f} s")

1,000,000 filas
pandas: 0.149 s
polars: 0.030 s


## 7. Convertir entre Pandas, Polars y NumPy

Scikit-learn acepta Pandas y Polars directamente; PyTorch necesita arrays de NumPy (o tensores).

In [14]:
de_polars_a_pandas = df_pl_limpio.to_pandas()
de_pandas_a_polars = pl.from_pandas(df_pd_limpio)
matriz = df_pl_limpio.select("metros", "habitaciones").to_numpy()
print(type(de_polars_a_pandas), type(de_pandas_a_polars), matriz.shape, matriz.dtype)

<class 'pandas.DataFrame'> <class 'polars.dataframe.frame.DataFrame'> (2000, 2) float64


## Chuleta Pandas ↔ Polars

| Tarea | Pandas | Polars |
|---|---|---|
| Leer CSV | `pd.read_csv(p)` | `pl.read_csv(p)` / `pl.scan_csv(p)` |
| Seleccionar | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filtrar | `df[df.a > 0]` / `df.query("a > 0")` | `df.filter(pl.col("a") > 0)` |
| Nueva columna | `df.assign(c=df.a * 2)` | `df.with_columns((pl.col("a") * 2).alias("c"))` |
| Agrupar | `df.groupby("k").agg(m=("a", "mean"))` | `df.group_by("k").agg(pl.col("a").mean())` |
| Faltantes | `df.isna().sum()` / `fillna` | `df.null_count()` / `fill_null` |
| Ordenar | `df.sort_values("a")` | `df.sort("a")` |

## ✏️ Ejercicios
1. Calcula, con ambas librerías, el precio medio por número de `habitaciones` y por `barrio` a la vez.
2. Crea una columna `categoria_antiguedad` (`"nueva"` < 10 años, `"media"` < 30, `"antigua"` resto).
   Pista Polars: `pl.when(...).then(...).otherwise(...)`.
3. Con `scan_csv`, obtén las 5 casas más caras de cada barrio (pista: `.sort(...).head(5).over(...)` o `group_by(...).head(5)`).